In [1]:
# %% 
# ================================
# STEP 1: OVERVIEW & IMPORTS
# Description:
# This script calculates half-hourly hot water load (W) for ALL buildings
# in the baseline_models folder.
#
# It:
# 1. Loads EPC data to get number of occupants for each building
# 2. Iterates through every building folder
# 3. Loads efficiency + ground temperature profiles
# 4. Simulates yearly hot water demand
# 5. Simulates tank behaviour (dynamic efficiency)
# 6. Saves a CSV (17,520 rows) into each building's HOT_WATER folder
# ================================

import numpy as np
import pandas as pd
import random
import os
from datetime import datetime, timedelta

In [ ]:
# %%
# ================================
# STEP 2: PATHS & BUILDING LIST
# Description: Define file paths and detect all building IDs automatically
# ================================

epc_path = "/Users/jakegehrung/Desktop/data_raw/data_processed/fintry_epc_NPV_IRR_update.csv"
baseline_path = "/Users/jakegehrung/Desktop/data_raw/baseline_models"
ground_temp_path = "/Users/jakegehrung/Desktop/data_raw/REFERENCE FILES/half_hourly_ground_temp.csv"

# Get all building folders
building_ids = [f for f in os.listdir(baseline_path) if f.isdigit()]

print(f"Total buildings found: {len(building_ids)}")
print("Sample:", building_ids[:5])

Total buildings found: 117
Sample: ['1001664924', '1001829085', '1001063639', '1001829071', '1234761002']


In [3]:
# %%
# ================================
# STEP 3: LOAD STATIC DATA
# Description: Load EPC and ground temperature once (used for all buildings)
# ================================

epc_df = pd.read_csv(epc_path)
gt_df = pd.read_csv(ground_temp_path)
ground_temp_profile = gt_df["ground_temp_30mm"].values

In [4]:
# %%
# ================================
# STEP 4: TIME VECTOR
# Description: Generate half-hourly timestamps for full year
# ================================

times = []
current = datetime(2026, 1, 1, 0, 30)

for i in range(17520):
    times.append(current.strftime("%H:%M:%S"))
    current += timedelta(minutes=30)

In [5]:
# %%
# ================================
# STEP 5: PROBABILITY PROFILES
# Description: Define usage probability curves
# ================================

p1 = [0.2,0.1,0.05,0.03,0.02,0.02,0.02,0.02,0.02,0.03,
      0.05,0.08,0.2,0.45,0.6,0.75,0.65,0.7,0.55,0.45,
      0.5,0.45,0.4,0.35,0.4,0.35,0.3,0.25,0.22,0.2,
      0.25,0.3,0.35,0.4,0.45,0.48,0.42,0.45,0.35,0.3,
      0.25,0.3,0.35,0.25,0.2,0.15,0.1,0.15]

p2 = [0.1,0.09,0.075,0.06,0.05,0.04,0.03,0.025,0.03,0.04,
      0.06,0.15,0.3,0.45,0.6,0.925,0.65,0.725,0.55,0.475,
      0.45,0.475,0.425,0.4,0.35,0.3,0.275,0.25,0.225,0.225,
      0.25,0.275,0.325,0.4,0.525,0.55,0.625,0.55,0.5,0.4,
      0.35,0.3,0.275,0.25,0.3,0.225,0.15,0.125]

p3 = [0.1,0.083333,0.066667,0.1,0.116667,0.1,0.083333,0.083333,0.066667,0.1,
      0.116667,0.166667,0.266667,0.366667,0.5,0.4,0.333333,0.383333,0.283333,0.25,
      0.233333,0.216667,0.2,0.183333,0.2,0.183333,0.166667,0.16,0.166667,0.173333,
      0.166667,0.183333,0.233333,0.333333,0.5,0.433333,0.466667,0.4,0.333333,0.266667,
      0.2,0.166667,0.25,0.216667,0.233333,0.25,0.216667,0.133333]

p4 = [0.05,0.025,0.02,0.0125,0.0125,0.0125,0.0125,0.0125,0.025,0.05,
      0.075,0.2,0.375,0.625,0.8,0.875,0.525,0.475,0.5,0.45,
      0.375,0.325,0.275,0.25,0.225,0.2125,0.225,0.2,0.1875,0.175,
      0.1875,0.2125,0.25,0.325,0.425,0.5,0.65,0.55,0.625,0.55,
      0.45,0.375,0.3,0.25,0.2,0.15,0.1,0.075]

In [6]:
# %%
# ================================
# STEP 6: SIMULATION FUNCTIONS
# Description: Define all simulation functions (unchanged logic)
# ================================

def get_probs(people):
    if people < 1.5:
        return p1
    elif people < 2.5:
        return p2
    elif people < 3.5:
        return p3
    else:
        return p4


def simulate_day(num_people, probs, weekend=False):
    showers_done = [0] * num_people
    medium_done = [0] * num_people

    MEDIUM_LOAD = 10 if weekend else 7.5

    preferences = [
        "morning" if random.random() < 0.7 else "evening"
        for _ in range(num_people)
    ]

    ltot = []

    for count in range(48):
        step_total = 0

        for person in range(num_people):
            r = random.random()
            load = 0

            morning_range = range(11, 17) if not weekend else range(15, 21)
            evening_range = range(34, 39)

            if count in morning_range:
                if (preferences[person] == "morning" and
                    not showers_done[person] and r < probs[count]):
                    load = 35
                    showers_done[person] = 1
                elif r < probs[count] * 0.3:
                    load = 0.5

            elif 17 <= count <= 33:
                if r < probs[count] * 0.2:
                    load = 0.5

            elif count in evening_range:
                if (preferences[person] == "evening" and
                    not showers_done[person] and r < probs[count]):
                    load = 35
                    showers_done[person] = 1
                elif (not medium_done[person] and r < probs[count]):
                    load = MEDIUM_LOAD
                    medium_done[person] = 1
                elif r < probs[count] * 0.3:
                    load = 0.5

            step_total += load

        ltot.append(step_total)

    return np.array(ltot)


def simulate_year(num_people, probs):
    start_date = datetime(2026, 1, 1)
    end_date = datetime(2026, 12, 31)

    current_date = start_date
    ltot_year = []

    while current_date <= end_date:
        weekend = current_date.weekday() >= 5

        multiplier = 1.2 if (
            (current_date.month == 12 and current_date.day >= 23) or
            (current_date.month == 1 and current_date.day <= 2)
        ) else 1.0

        day_litres = simulate_day(num_people, probs, weekend)
        ltot_year.extend(day_litres * multiplier)

        current_date += timedelta(days=1)

    return np.array(ltot_year)


def simulate_tank_dynamic(ltot_year, people, eff_profile, ground_temp_profile):

    tank_volume = 135 if people <= 2 else 195

    if people < 3:
        heater_power = 8000
    else:
        heater_power = 13000

    timestep_hours = 0.5
    tank_temp = 60

    tank_energy = tank_volume * 4180 * (tank_temp - ground_temp_profile[0]) / 3600000
    power_profile = []

    for i in range(len(ltot_year)):

        litres_used = ltot_year[i]
        Tg = ground_temp_profile[i]
        eff = eff_profile[i]

        hour = (i % 48) / 2

        max_energy = tank_volume * 4180 * (tank_temp - Tg) / 3600000

        # -----------------------
        # DEMAND (independent)
        # -----------------------
        energy_used = (4180 * (tank_temp - Tg) * litres_used) / 3600000
        tank_energy -= energy_used

        # -----------------------
        # LOSSES
        # -----------------------
        tank_energy -= 0.08 * timestep_hours
        tank_energy = max(tank_energy, 0)

        # -----------------------
        # CONTROL (NOT efficiency-based)
        # -----------------------
        reheat_threshold = 0.8 * max_energy
        is_night = (hour >= 22) or (hour <= 5)

        if is_night and tank_energy < reheat_threshold:
            energy_needed = max_energy - tank_energy
            max_boiler_energy = heater_power * timestep_hours / 1000

            energy_added = min(max_boiler_energy, energy_needed)

            tank_energy += energy_added

            thermal_power = (energy_added * 1000) / timestep_hours
            boiler_input_power = thermal_power / eff  # ✅ correct use of efficiency

        else:
            boiler_input_power = 0

        power_profile.append(boiler_input_power)

    return np.array(power_profile)

In [7]:
# %%
# ================================
# STEP 7: MAIN LOOP OVER BUILDINGS
# Description: Run full simulation for every building folder
# ================================

for building_id in building_ids:
    try:
        print(f"\nProcessing building: {building_id}")

        # Get occupants
        people = int(
            epc_df.loc[
                epc_df["BUILDING_REFERENCE_NUMBER"] == int(building_id),
                "OCCUPANTS_ROUNDED_UP"
            ].values[0]
        )

        probs = get_probs(people)

        # Paths
        eff_path = f"{baseline_path}/{building_id}/HOT_WATER/water_heat_efficiency.csv"
        output_path = f"{baseline_path}/{building_id}/HOT_WATER/hot_water_load.csv"

        # Load efficiency
        eff_df = pd.read_csv(eff_path)
        eff_profile = eff_df["water_heat_efficiency"].values

        # Simulate
        ltot_year = simulate_year(people, probs)
        watts_year = simulate_tank_dynamic(
            ltot_year, people, eff_profile, ground_temp_profile
        )

        # Save
        df = pd.DataFrame({
            "Time": times,
            "hot_water_load": watts_year
        })

        df.to_csv(output_path, index=False)

        print(f"Saved: {output_path}")

    except Exception as e:
        print(f"Skipped {building_id} due to error: {e}")


Processing building: 1001664924
Saved: /Users/jakegehrung/Desktop/data_raw/baseline_models/1001664924/HOT_WATER/hot_water_load.csv

Processing building: 1001829085
Saved: /Users/jakegehrung/Desktop/data_raw/baseline_models/1001829085/HOT_WATER/hot_water_load.csv

Processing building: 1001063639
Saved: /Users/jakegehrung/Desktop/data_raw/baseline_models/1001063639/HOT_WATER/hot_water_load.csv

Processing building: 1001829071
Saved: /Users/jakegehrung/Desktop/data_raw/baseline_models/1001829071/HOT_WATER/hot_water_load.csv

Processing building: 1234761002
Saved: /Users/jakegehrung/Desktop/data_raw/baseline_models/1234761002/HOT_WATER/hot_water_load.csv

Processing building: 1002539407
Saved: /Users/jakegehrung/Desktop/data_raw/baseline_models/1002539407/HOT_WATER/hot_water_load.csv

Processing building: 1001063637
Saved: /Users/jakegehrung/Desktop/data_raw/baseline_models/1001063637/HOT_WATER/hot_water_load.csv

Processing building: 1001664941
Saved: /Users/jakegehrung/Desktop/data_raw/